# Notebook for å autentisere seg mot skyporten og hente ut data fra share

In [9]:
! pip install -r requirements.txt

In [12]:
import os
import json
import subprocess
import delta_sharing
from google.cloud import storage

from src.auth import generate_access_token
from src.utils import (
    fetch_config_share,
    create_credentials_config,
)

ImportError: cannot import name 'fetch_config_share' from 'src.utils' (unknown location)

In [ ]:

project_id = "innsikt-data-dev-ec28"
project_num = "614733074632"
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-dev/providers/skyporten-bi-provider-dev"
random_id = "9ivj"
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-dev"

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
CONFIG_PATH = "configs/config.json"
SOURCE_PATH = "share/config.share" #filen du leser fra dersom du har en gyldig config.share, eller ønsker å skrive til dersom share har utløpt

In [ ]:
create_credentials_config(provider_full_identifier, TOKEN_PATH, CREDENTIALS_PATH)

In [ ]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)



In [ ]:
os.makedirs("share", exist_ok=True)
bucket_id = f"sp-{project_id}-{random_id}"
fetch_config_share(bucket_id, SOURCE_PATH, CREDENTIALS_PATH)

In [ ]:
sharing_client = delta_sharing.SharingClient(SOURCE_PATH)

tables = []
try:
    tables = sharing_client.list_all_tables()
except:
    print("Sharen har ikke tilgang til noen tabeller")
    exit(1)

for table in tables:
    print(table.name)

In [ ]:
table_name = tables[0].name #henter første tabell som er tilgjengelig i share, erstatt med ønsket tabellnavn 
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
data = delta_sharing.load_as_pandas(table_url)
print(data.head())

In [ ]:
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
metadata = delta_sharing.get_table_metadata(table_url)

print(metadata)